<a href="https://colab.research.google.com/github/mithunaananthan10-bot/daily_habit/blob/main/Carpriceprediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===========================
# CAR PRICE PREDICTION USING RANDOM FOREST
# ===========================

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
df = pd.read_csv("quikr_car.csv")

df = df[df["Price"] != "Ask For Price"]
df["Price"] = df["Price"].str.replace(",", "", regex=False)
df["Price"] = df["Price"].astype(int)

# Keep only numeric years
df = df[df["year"].astype(str).str.isdigit()]
df["year"] = df["year"].astype(int)

df = df[df["kms_driven"] != "Petrol"]
df["kms_driven"] = df["kms_driven"].str.replace(",", "", regex=False)
df["kms_driven"] = df["kms_driven"].str.replace(" kms", "", regex=False)
df["kms_driven"] = df["kms_driven"].astype(int)

# Remove missing values
df.dropna(inplace=True)

# Remove unrealistic prices (Outliers)
df = df[df["Price"] < 6000000]
X = df.drop("Price", axis=1)
Y = df["Price"]

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42
)
transformer = ColumnTransformer(
    transformers=[
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore"),
            ["company", "name", "fuel_type"]
        )
    ],
    remainder="passthrough"
)

model = Pipeline([
    ("preprocessor", transformer),
    ("regressor", RandomForestRegressor(
        n_estimators=300,
        random_state=42
    ))
])
model.fit(X_train, Y_train)
accuracy = model.score(X_test, Y_test)

print("\nModel Accuracy :", round(accuracy * 100, 2), "%")

# ---------------------------
# User Input
# ---------------------------

print("\n==============================")
print("      CAR PRICE PREDICTOR")
print("==============================")

company = input("Enter Company      : ")
name = input("Enter Car Name     : ")
year = int(input("Enter Year         : "))
fuel_type = input("Enter Fuel Type    : ")
kms_driven = int(input("Enter KM Driven    : "))

new_car = pd.DataFrame({
    "company": [company],
    "name": [name],
    "year": [year],
    "kms_driven": [kms_driven],
    "fuel_type": [fuel_type]
})

# ---------------------------
# Prediction
# ---------------------------

prediction = model.predict(new_car)

print("\n----------------------------")
print("Predicted Price : ₹{:,.0f}".format(prediction[0]))
print("Model Accuracy  : {:.2f}%".format(accuracy * 100))
print("----------------------------")


Model Accuracy : 53.33 %

      CAR PRICE PREDICTOR
Enter Company      : Ford
Enter Car Name     : Ford Figo
Enter Year         : 2020
Enter Fuel Type    : Diesel
Enter KM Driven    : 44000

----------------------------
Predicted Price : ₹641,987
Model Accuracy  : 53.33%
----------------------------
